In [1]:
import csv
import os

# Project data directory
DATA_DIR = "../data"

SOIL_FILE = os.path.join(DATA_DIR, "soil.csv")
CROP_FILE = os.path.join(DATA_DIR, "crops.csv")
OUTPUT_FILE = os.path.join(DATA_DIR, "soil_crop_suitability_v2.csv")


# Convert qualitative soil pH categories into approximate ranges.
# These are rough assumptions, not actual soil measurements.
PH_CATEGORY_RANGES = {
    "very strongly acidic": (3.5, 4.5),
    "strongly acidic": (4.0, 5.0),
    "moderately acidic": (4.5, 5.5),
    "slightly acidic": (5.5, 6.5),
    "slightly acidic to neutral": (5.5, 7.0),
    "neutral": (6.5, 7.5),
    "near neutral": (6.0, 7.5),
    "alkaline": (7.5, 8.5),
}


def score_texture(soil_texture, preferred_texture):
    """Estimate soil texture compatibility."""
    soil = soil_texture.lower()
    preferred = preferred_texture.lower()

    if preferred in soil or soil in preferred:
        return 100

    soil_terms = set(soil.replace("-", " ").split())
    preferred_terms = set(preferred.replace("-", " ").split())

    if soil_terms.intersection(preferred_terms):
        return 70

    return 40


def score_drainage(soil_drainage, preferred_drainage):
    """Estimate drainage compatibility."""
    soil = soil_drainage.strip().lower()
    preferred = preferred_drainage.strip().lower()

    if soil == preferred:
        return 100

    # A different drainage category does not necessarily mean
    # a crop is unsuitable, so assign a partial score.
    return 40


def score_ph(ph_category, preferred_ph_range):
    """
    Estimate pH compatibility using approximate category ranges.
    Returns (score, status).
    """
    category = ph_category.strip().lower()

    if category not in PH_CATEGORY_RANGES:
        return "", "Unknown soil pH category"

    try:
        crop_min, crop_max = map(
            float, preferred_ph_range.split("-")
        )
    except (ValueError, AttributeError):
        return "", "Invalid crop pH range"

    soil_min, soil_max = PH_CATEGORY_RANGES[category]

    # Calculate overlap between the estimated soil pH interval
    # and the crop's preferred pH interval.
    overlap = max(
        0,
        min(soil_max, crop_max) - max(soil_min, crop_min)
    )

    soil_width = soil_max - soil_min

    if soil_width <= 0:
        return "", "Invalid soil pH interval"

    score = round((overlap / soil_width) * 100, 2)

    return score, "Estimated from qualitative pH category"


# Read soil records
with open(SOIL_FILE, "r", newline="", encoding="utf-8-sig") as file:
    soils = list(csv.DictReader(file))

# Read crop records
with open(CROP_FILE, "r", newline="", encoding="utf-8-sig") as file:
    crops = list(csv.DictReader(file))


results = []

for soil in soils:
    for crop in crops:

        texture_score = score_texture(
            soil.get("soil_texture", ""),
            crop.get("preferred_soil_texture", "")
        )

        drainage_score = score_drainage(
            soil.get("drainage", ""),
            crop.get("preferred_drainage", "")
        )

        ph_score, ph_status = score_ph(
            soil.get("ph_category", ""),
            crop.get("preferred_ph_range", "")
        )

        # If pH cannot be estimated, do not silently treat it as zero.
        if ph_score == "":
            suitability_score = round(
                (texture_score + drainage_score) / 2, 2
            )
            score_method = "Texture + drainage only; pH unavailable"
        else:
            suitability_score = round(
                (texture_score + drainage_score + ph_score) / 3, 2
            )
            score_method = "Equal-weight heuristic; pH estimated"

        results.append({
            "district": soil.get("district", ""),
            "soil_type": soil.get("soil_type", ""),
            "crop": crop.get("crop", ""),
            "texture_score": texture_score,
            "drainage_score": drainage_score,
            "estimated_ph_score": ph_score,
            "suitability_score": suitability_score,
            "ph_status": ph_status,
            "score_method": score_method
        })


# Save results
with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as file:
    fieldnames = [
        "district",
        "soil_type",
        "crop",
        "texture_score",
        "drainage_score",
        "estimated_ph_score",
        "suitability_score",
        "ph_status",
        "score_method"
    ]

    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)


print("Improved suitability dataset created!")
print("Total comparisons:", len(results))
print("Output file:", OUTPUT_FILE)

print("\nFirst 10 results:")

for row in results[:10]:
    print(
        row["district"],
        "|", row["soil_type"],
        "|", row["crop"],
        "| Score:", row["suitability_score"]
    )

Improved suitability dataset created!
Total comparisons: 230
Output file: ../data\soil_crop_suitability_v2.csv

First 10 results:
Thiruvananthapuram | Red soil | Rice | Score: 48.89
Thiruvananthapuram | Red soil | Coconut | Score: 90.0
Thiruvananthapuram | Red soil | Banana | Score: 100.0
Thiruvananthapuram | Red soil | Black pepper | Score: 88.89
Thiruvananthapuram | Red soil | Rubber | Score: 77.78
Thiruvananthapuram | Red soil | Tapioca | Score: 90.0
Thiruvananthapuram | Red soil | Ginger | Score: 88.89
Thiruvananthapuram | Red soil | Turmeric | Score: 100.0
Thiruvananthapuram | Red soil | Groundnut | Score: 90.0
Thiruvananthapuram | Red soil | Vegetables | Score: 100.0
